# Tipos de datos faltantes

## 1. MCAR (Missing Completely At Random)

Un dato es **MCAR** cuando falta de manera completamente aleatoria, sin relación con ninguna variable del dataset, ni observada ni no observada.

### Idea clave
La ausencia ocurre al azar puro.

### Ejemplo conceptual
Si durante el registro de pesos de pacientes, algunas filas se pierden porque hubo un corte eléctrico aleatorio, esos faltantes podrían ser MCAR.

### Características
- no depende de otras columnas
- no depende del valor propio faltante
- es el caso menos problemático
- si eliminamos filas, el sesgo suele ser menor

---

## 2. MAR (Missing At Random)

Un dato es **MAR** cuando la ausencia sí depende de otras variables observadas en el dataset, pero no del valor faltante en sí mismo.

### Idea clave
El faltante se puede explicar por otras columnas que sí conocemos.

### Ejemplo conceptual
Supongamos que la variable `salario` falta más frecuentemente en personas jóvenes que en personas mayores.  
Aquí la falta depende de la variable observada `edad`.

### Características
- la ausencia está relacionada con variables observadas
- no depende directamente del valor faltante
- requiere análisis cuidadoso
- técnicas como imputación por grupos o modelos suelen ser útiles

---

## 3. MNAR (Missing Not At Random)

Un dato es **MNAR** cuando la ausencia depende del propio valor que falta o de una causa no observada.

### Idea clave
El hecho de que falte el dato está relacionado con el mismo dato ausente.

### Ejemplo conceptual
En una encuesta de ingresos, las personas con ingresos muy altos pueden preferir no responder.  
Entonces la falta de `ingreso` depende del valor real del ingreso.

### Características
- es el caso más difícil de tratar
- puede introducir sesgo fuerte
- imputar sin entender el contexto puede ser peligroso
- muchas veces se necesita conocimiento del negocio o análisis adicional



In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

n = 200

df = pd.DataFrame({
    "edad": np.random.randint(18, 65, size=n),
    "horas_estudio": np.random.normal(5, 2, size=n).round(1),
    "ingreso": np.random.normal(2500, 800, size=n).round(0),
})

# evitamos horas negativas
df["horas_estudio"] = df["horas_estudio"].clip(lower=0.5)

df.head()

,edad,horas_estudio,ingreso
0,56,5.0,3742.0
1,46,1.7,2148.0
2,32,2.9,2174.0
3,60,3.0,2709.0
4,25,5.2,2510.0


In [2]:
df.isna().sum()

,0
edad,0
horas_estudio,0
ingreso,0


# simular MCAR

In [3]:
df_mcar = df.copy()

# seleccionamos filas aleatorias
indices_mcar = np.random.choice(df_mcar.index, size=30, replace=False)
df_mcar.loc[indices_mcar, "horas_estudio"] = np.nan

print("Cantidad de faltantes por columna:")
print(df_mcar.isna().sum())

df_mcar.head(10)

Cantidad de faltantes por columna:
edad              0
horas_estudio    30
ingreso           0
dtype: int64


,edad,horas_estudio,ingreso
0,56,5.0,3742.0
1,46,1.7,2148.0
2,32,2.9,2174.0
3,60,3.0,2709.0
4,25,5.2,2510.0
5,38,4.1,3245.0
6,56,3.7,2081.0
7,36,5.0,3259.0
8,40,6.0,2309.0
9,28,4.5,3183.0


# comprobar intuitivamente MCAR

In [4]:
print("Promedio de edad cuando horas_estudio NO falta:")
print(df_mcar[df_mcar["horas_estudio"].notna()]["edad"].mean())

print("\nPromedio de edad cuando horas_estudio SÍ falta:")
print(df_mcar[df_mcar["horas_estudio"].isna()]["edad"].mean())

Promedio de edad cuando horas_estudio NO falta:
41.95882352941177

Promedio de edad cuando horas_estudio SÍ falta:
39.36666666666667


# NOTA : Si los faltantes son realmente aleatorios, no debería haber una diferencia sistemática fuerte entre ambos grupos.

# simular MAR

In [5]:
df_mar = df.copy()

# probabilidad de faltar según edad
prob_faltante = np.where(df_mar["edad"] < 30, 0.45, 0.10)

random_vals = np.random.rand(len(df_mar))
df_mar.loc[random_vals < prob_faltante, "ingreso"] = np.nan

print("Cantidad de faltantes por columna:")
print(df_mar.isna().sum())

df_mar.head(10)

Cantidad de faltantes por columna:
edad              0
horas_estudio     0
ingreso          29
dtype: int64


,edad,horas_estudio,ingreso
0,56,5.0,NaN
1,46,1.7,NaN
2,32,2.9,2174.0
3,60,3.0,2709.0
4,25,5.2,2510.0
5,38,4.1,3245.0
6,56,3.7,2081.0
7,36,5.0,3259.0
8,40,6.0,2309.0
9,28,4.5,3183.0


## Aquí el faltante en ingreso no ocurre por azar total.
Ocurre más en personas jóvenes, por lo que depende de una variable observada.

## comprobar patrón MAR

In [6]:
faltante_ingreso = df_mar["ingreso"].isna()

print("Edad promedio cuando ingreso NO falta:")
print(df_mar.loc[~faltante_ingreso, "edad"].mean())

print("\nEdad promedio cuando ingreso SÍ falta:")
print(df_mar.loc[faltante_ingreso, "edad"].mean())

Edad promedio cuando ingreso NO falta:
43.251461988304094

Edad promedio cuando ingreso SÍ falta:
31.655172413793103


## NOTA : Si el grupo con faltantes tiene una edad promedio claramente distinta, eso apoya la idea de MAR.

# simular MNAR

In [7]:
df_mnar = df.copy()

# mayor probabilidad de faltar si el ingreso es alto
prob_faltante_mnar = np.where(df_mnar["ingreso"] > 3200, 0.50, 0.08)

random_vals = np.random.rand(len(df_mnar))
df_mnar.loc[random_vals < prob_faltante_mnar, "ingreso"] = np.nan

print("Cantidad de faltantes por columna:")
print(df_mnar.isna().sum())

df_mnar.head(10)

Cantidad de faltantes por columna:
edad              0
horas_estudio     0
ingreso          38
dtype: int64


,edad,horas_estudio,ingreso
0,56,5.0,3742.0
1,46,1.7,2148.0
2,32,2.9,2174.0
3,60,3.0,2709.0
4,25,5.2,2510.0
5,38,4.1,3245.0
6,56,3.7,2081.0
7,36,5.0,3259.0
8,40,6.0,NaN
9,28,4.5,3183.0


## Este es un caso tipo encuesta donde quienes ganan más podrían no querer reportar su ingreso.
La ausencia está relacionada con el propio valor faltante.

In [8]:
mask_missing = df_mnar["ingreso"].isna()

print("Ingreso real promedio en filas que terminaron faltando:")
print(df.loc[mask_missing, "ingreso"].mean())

print("\nIngreso real promedio en filas que NO faltaron:")
print(df.loc[~mask_missing, "ingreso"].mean())

Ingreso real promedio en filas que terminaron faltando:
2755.4210526315787

Ingreso real promedio en filas que NO faltaron:
2498.8086419753085


Aquí usamos el dataset original para mostrar que justo las filas con mayores ingresos fueron las que más tendieron a desaparecer.
Eso representa un patrón MNAR.